In [1]:
import re
import pandas as pd
import yfinance as yf

In [23]:
# ==============================================================================
# 1. INPUT TANGGAL EVALUASI & RAW TEXT TRADING PLAN
# ==============================================================================
tanggal_evaluasi = "2026-09-18"  # Tanggal Acuan Evaluasi / Target Eksekusi (H+0)
JUMLAH_HARI_LOOKAHEAD = 10      # Evaluasi rentang H+0 s/d H+10 hari kalender

In [24]:
# TEMPEL SELURUH TEKS TRADING PLAN DI SINI
raw_text = """
==================================================
 TRADING PLAN: AISA.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-18 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-21 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp120
. Cut Loss (-3%)    : Rp116
. Trailing Trigger  : Rp128 (+7%)
. Trailing Lock     : Rp126 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: MAPI.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-18 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-21 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp1,500
. Cut Loss (-3%)    : Rp1,455
. Trailing Trigger  : Rp1,605 (+7%)
. Trailing Lock     : Rp1,575 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: ULTJ.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-18 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-21 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp2,060
. Cut Loss (-3%)    : Rp1,998
. Trailing Trigger  : Rp2,204 (+7%)
. Trailing Lock     : Rp2,163 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: NASI.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-18 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-21 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp234
. Cut Loss (-3%)    : Rp227
. Trailing Trigger  : Rp250 (+7%)
. Trailing Lock     : Rp246 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: JELI.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-18 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-21 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp500
. Cut Loss (-3%)    : Rp485
. Trailing Trigger  : Rp535 (+7%)
. Trailing Lock     : Rp525 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: AGII.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-18 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-21 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp3,740
. Cut Loss (-3%)    : Rp3,628
. Trailing Trigger  : Rp4,002 (+7%)
. Trailing Lock     : Rp3,927 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: BAJA.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-18 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-21 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp350
. Cut Loss (-3%)    : Rp340
. Trailing Trigger  : Rp374 (+7%)
. Trailing Lock     : Rp368 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================




"""


In [25]:
# ==============================================================================
# 2. PARSER TRADING PLAN (ANTI-GAGAL SPASI)
# ==============================================================================
def parse_plans(text):
    cleaned_text = text.replace('\xa0', ' ')
    blocks = cleaned_text.split("TRADING PLAN:")
    plans = []

    for block in blocks[1:]:
        ticker = re.search(r"^\s*([A-Za-z0-9\.]+)", block)
        buy = re.search(r"Harga Beli\s*Acuan\s*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)
        cut_loss = re.search(r"Cut Loss[^\n:]*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)
        lock = re.search(r"Trailing Lock[^\n:]*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)

        if ticker and buy and cut_loss and lock:
            code_raw = ticker.group(1).strip()
            ticker_formatted = code_raw if code_raw.endswith(".JK") else f"{code_raw}.JK"

            plans.append({
                "ticker": ticker_formatted,
                "code": ticker_formatted.replace(".JK", ""),
                "buy": int(buy.group(1).replace(",", "").replace(".", "")),
                "cut_loss": int(cut_loss.group(1).replace(",", "").replace(".", "")),
                "lock": int(lock.group(1).replace(",", "").replace(".", ""))
            })
    return plans

parsed_data = parse_plans(raw_text)

if not parsed_data:
    print("X Teks gagal dibaca. Pastikan format teks sesuai.")
else:
    df = pd.DataFrame(parsed_data)
    tickers = df["ticker"].tolist()

In [26]:
# ==============================================================================
# 3. AMBIL DATA DARI TANGGAL EVALUASI (H+0 s/d H+N)
# ==============================================================================
t_start = pd.to_datetime(tanggal_evaluasi)
t_end = t_start + pd.Timedelta(days=JUMLAH_HARI_LOOKAHEAD)

start_str = t_start.strftime('%Y-%m-%d')
end_str = t_end.strftime('%Y-%m-%d')

print(f"Membaca {len(tickers)} saham... Mengambil data dari {start_str} s/d {end_str}...")

# Download batch data historis
downloaded_data = yf.download(
    tickers,
    start=start_str,
    end=end_str,
    group_by='ticker',
    auto_adjust=False,
    progress=False
)

highest_high_list = []
lowest_low_list = []
tgl_open_list = []
status_list = []
tgl_tereksekusi_list = []

for idx, row in df.iterrows():
    ticker = row["ticker"]

    try:
        # Mengambil DataFrame spesifik per ticker
        df_tk = downloaded_data if len(tickers) == 1 else downloaded_data[ticker].dropna()

        # Jika ada MultiIndex level pada kolom
        if isinstance(df_tk.columns, pd.MultiIndex):
            df_tk.columns = df_tk.columns.get_level_values(0)

        if df_tk.empty:
            highest_high_list.append(None)
            lowest_low_list.append(None)
            tgl_open_list.append("-")
            status_list.append("DATA TIDAK ADA")
            tgl_tereksekusi_list.append("-")
            continue

        # Rekam High tertinggi & Low terendah selama periode
        highest_high_list.append(df_tk["High"].max())
        lowest_low_list.append(df_tk["Low"].min())

        # ==============================================================================
        # 4. CEK TANGGAL POSISI OPEN & TANGGAL EVALUASI TEREKSEKUSI
        # ==============================================================================
        tgl_open = "-"
        status = "HOLD"
        tgl_kejadian = "-"
        is_opened = False

        for date, price_row in df_tk.iterrows():
            date_str = date.strftime('%Y-%m-%d')

            # Cek kapan posisi Open pertama kali terjadi (Harga Low <= Harga Beli Acuan)
            if not is_opened and price_row["Low"] <= row["buy"]:
                tgl_open = date_str
                is_opened = True

            # Jika sudah pernah ter-open, cek status Profit / Loss
            if is_opened:
                # Prioritas 1: Menyentuh/Melewati Target Profit
                if price_row["High"] >= row["lock"]:
                    status = "PROFIT"
                    tgl_kejadian = date_str
                    break
                # Prioritas 2: Menyentuh/Turun ke Cut Loss
                elif price_row["Low"] <= row["cut_loss"]:
                    status = "LOSS"
                    tgl_kejadian = date_str
                    break

        tgl_open_list.append(tgl_open)
        status_list.append(status if is_opened else "BELUM OPEN")
        tgl_tereksekusi_list.append(tgl_kejadian)

    except Exception:
        highest_high_list.append(None)
        lowest_low_list.append(None)
        tgl_open_list.append("-")
        status_list.append("ERROR")
        tgl_tereksekusi_list.append("-")

# Masukkan hasil ke DataFrame
df["Highest High"] = highest_high_list
df["Lowest Low"] = lowest_low_list
df["Tanggal Posisi Open"] = tgl_open_list
df["Status Evaluasi"] = status_list
df["Tanggal Tereksekusi"] = tgl_tereksekusi_list

# ==============================================================================
# 5. TAMPILKAN HASIL EVALUASI
# ==============================================================================
print("\n=== HASIL EVALUASI TRADING PLAN ===")
cols_to_display = [
    "code", "buy", "cut_loss", "lock",
    "Highest High", "Lowest Low",
    "Tanggal Posisi Open", "Status Evaluasi", "Tanggal Tereksekusi"
]
display(df[cols_to_display])

Membaca 7 saham... Mengambil data dari 2026-09-18 s/d 2026-09-28...

=== HASIL EVALUASI TRADING PLAN ===


,code,buy,cut_loss,lock,Highest High,Lowest Low,Tanggal Posisi Open,Status Evaluasi,Tanggal Tereksekusi
0,AISA,120,116,126,146.0,115.0,2026-09-18,PROFIT,2026-09-18
1,MAPI,1500,1455,1575,1515.0,1445.0,2026-09-18,LOSS,2026-09-22
2,ULTJ,2060,1998,2163,2190.0,1885.0,2026-09-18,PROFIT,2026-09-18
3,NASI,234,227,246,364.0,174.0,2026-09-18,LOSS,2026-09-18
4,JELI,500,485,525,545.0,438.0,2026-09-18,PROFIT,2026-09-18
5,AGII,3740,3628,3927,3800.0,3300.0,2026-09-18,LOSS,2026-09-18
6,BAJA,350,340,368,505.0,276.0,2026-09-18,LOSS,2026-09-18
